# Frankie ComfyUI Server — Exposed via Cloudflared Tunnel

**Purpose:** boot ComfyUI on Colab T4, expose it as a public HTTPS endpoint so the Frankie Telegram bot can call it from anywhere.

**You do:** Runtime → T4 GPU → Run all → copy the URL printed at the bottom → ssh into mm and set it.

**Lifetime:** Colab disconnects after ~3hr idle / ~12hr max. While alive, `/selfie` routes here. When it dies, falls back to PiAPI Flux Kontext.

In [ ]:
!nvidia-smi | head -15

In [ ]:
import os
if not os.path.exists('/content/ComfyUI'):
    !git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
%cd /content/ComfyUI
!pip install -q -r requirements.txt
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus'):
    !git clone -q https://github.com/cubiq/ComfyUI_IPAdapter_plus /content/ComfyUI/custom_nodes/ComfyUI_IPAdapter_plus
!pip install -q insightface onnxruntime-gpu opencv-python-headless
print('ComfyUI ready')

In [ ]:
!mkdir -p /content/ComfyUI/models/checkpoints /content/ComfyUI/models/ipadapter /content/ComfyUI/models/loras /content/ComfyUI/models/clip_vision /content/ComfyUI/models/insightface/models/buffalo_l
CKPT = '/content/ComfyUI/models/checkpoints/realvis_xl_v5.safetensors'
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 1_000_000_000:
    !rm -f {CKPT}
    !wget --show-progress -O {CKPT} https://huggingface.co/SG161222/RealVisXL_V5.0/resolve/main/RealVisXL_V5.0_fp16.safetensors
if not os.path.exists('/content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin'):
    !wget -q --show-progress -O /content/ComfyUI/models/ipadapter/ip-adapter-faceid-plusv2_sdxl.bin https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl.bin
    !wget -q --show-progress -O /content/ComfyUI/models/loras/ip-adapter-faceid-plusv2_sdxl_lora.safetensors https://huggingface.co/h94/IP-Adapter-FaceID/resolve/main/ip-adapter-faceid-plusv2_sdxl_lora.safetensors
if not os.path.exists('/content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors'):
    !wget -q --show-progress -O /content/ComfyUI/models/clip_vision/CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors
if not os.path.exists('/content/ComfyUI/models/insightface/models/buffalo_l/det_10g.onnx'):
    !wget -q -O /tmp/buffalo_l.zip https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip
    !unzip -q -o /tmp/buffalo_l.zip -d /content/ComfyUI/models/insightface/models/buffalo_l
!ls -lh /content/ComfyUI/models/checkpoints

In [ ]:
# Stage both Frankie + Honey refs so workflows can reference either by filename
import requests
os.makedirs('/content/ComfyUI/input', exist_ok=True)
for name in ('frankie-ref.png', 'honey-ref.png'):
    url = f'https://raw.githubusercontent.com/veyrapay/frankie-assets/main/{name}'
    with open(f'/content/ComfyUI/input/{name}', 'wb') as f:
        f.write(requests.get(url).content)
    print(f'staged {name}')

In [ ]:
# Install cloudflared (for public tunnel)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Start ComfyUI server in background
!pkill -f 'main.py' 2>/dev/null; sleep 3
import subprocess, time, requests as r
subprocess.Popen(['python', '/content/ComfyUI/main.py', '--listen', '127.0.0.1', '--port', '8188', '--disable-auto-launch'], cwd='/content/ComfyUI')
for i in range(60):
    try:
        r.get('http://127.0.0.1:8188/system_stats', timeout=2)
        print(f'ComfyUI up after {i*2}s')
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('ComfyUI did not start')

In [ ]:
# Start cloudflared tunnel, parse public URL from stderr
import subprocess, re, time, os
tun_log = '/tmp/cloudflared.log'
open(tun_log, 'w').close()
!pkill -f cloudflared 2>/dev/null; sleep 1
subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8188'], stdout=open(tun_log, 'w'), stderr=subprocess.STDOUT)
public_url = None
for i in range(60):
    time.sleep(2)
    try:
        log = open(tun_log).read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
        if m:
            public_url = m.group(0)
            break
    except FileNotFoundError:
        pass
if not public_url:
    print('Tunnel did not come up in 120s. Last log:')
    print(open(tun_log).read()[-2000:])
    raise RuntimeError('cloudflared tunnel failed')
print()
print('=' * 70)
print('COMFY_URL =', public_url)
print('=' * 70)
print()
print('Wire to Frankie bot on mm with this one command:')
print()
print(f'  ssh mm "echo \\"export COMFY_URL=\'{public_url}\'\\" >> ~/.openclaw/workspace/.secrets/fal-ai.env && pkill -f frankie-selfie.mjs; source ~/.openclaw/workspace/.secrets/fal-ai.env && nohup env PIAPI_KEY=\\\$PIAPI_KEY COMFY_URL=\\\$COMFY_URL FRANKIE_BOT_TOKEN=\'REDACTED_ROTATE_VIA_BOTFATHER\' node ~/.openclaw/workspace/dispatchers/frankie-selfie.mjs > ~/.openclaw/logs/frankie-selfie.log 2>&1 & disown"')
print()
print('Then /selfie in Telegram routes to ComfyUI (RealVis XL + IPAdapter) instead of PiAPI.')

In [ ]:
# Keep notebook alive — Colab kicks idle sessions. This cell stays running until Stop.
import time
print('Server alive. Stop this cell to release the runtime.')
while True:
    time.sleep(60)
    log = open('/tmp/cloudflared.log').read()
    if 'error' in log.lower()[-1000:]:
        print('cloudflared log tail:')
        print(log[-500:])